In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Milestone - 2

In [2]:
# !pip install -q transformers datasets sentence-transformers scikit-learn torch

import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    pipeline,
)
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Loading Dataset

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

train = dataset["train"]

print(train)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})


### Question -1

In [3]:
def combine(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

train = train.map(combine)

print("Q1 Answer:")
print(len(train[51]["combined_text"]))

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Q1 Answer:
614


### Question - 2

In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("\nQ2 Answer:")
print(tokenizer.vocab_size)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Q2 Answer:
30522


### Question - 3 

In [5]:
print("\nQ3 Answer:")
print(tokenizer.sep_token_id)


Q3 Answer:
102


### Question - 4 

In [6]:
prompts = [str(x) if x is not None else "" for x in train["prompt"]]

encodings = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encodings["input_ids"].shape)

torch.Size([2000, 128])


### Question - 5

In [7]:
hidden_size = 768
num_heads = 12

print("\nQ5 Answer:")
print(hidden_size // num_heads)


Q5 Answer:
64


### Question - 6

In [8]:
model = AutoModel.from_pretrained("bert-base-uncased")

inputs = tokenizer(
    train[0]["prompt"],
    return_tensors="pt"
)

outputs = model(**inputs)

print("\nQ6 Answer:")
print(outputs.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Q6 Answer:
torch.Size([1, 31, 768])


### Question - 7

In [9]:
cls_embedding = outputs.last_hidden_state[0,0]

answer = cls_embedding[:5].sum().item()

print("\nQ7 Answer:")
print(round(answer,4))


Q7 Answer:
-1.2001


### Question - 8

In [10]:
model_attention = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

text = "Light-ion fusion is a technique."

inputs = tokenizer(
    text,
    return_tensors="pt"
)

outputs = model_attention(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("\nTokens:")
print(tokens)

fusion_index = tokens.index("fusion")

attention = outputs.attentions[-1]

# last layer
# first head
# CLS -> fusion

value = attention[0,0,0,fusion_index].item()

print("\nQ8 Answer:")
print(round(value,4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Tokens:
['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']

Q8 Answer:
0.1025


### Question - 9 

In [11]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

prompt_emb = embedder.encode(
    train[0]["prompt"],
    convert_to_tensor=True
)

optionB_emb = embedder.encode(
    train[0]["B"],
    convert_to_tensor=True
)

score = util.cos_sim(prompt_emb, optionB_emb)

print("\nQ9 Answer:")
print(round(score.item(),4))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Q9 Answer:
0.7658


### Question - 10

In [12]:
letters = ["A","B","C","D","E"]

def apk(actual,predicted,k=3):
    if actual in predicted[:k]:
        return 1/(predicted.index(actual)+1)
    return 0

# TF-IDF

tfidf_scores = []

tfidf_correct = []

mini_correct = []

for row in train:

    docs = [
        row["prompt"],
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]

    tfidf = TfidfVectorizer()

    X = tfidf.fit_transform(docs)

    sims = cosine_similarity(X[0],X[1:]).flatten()

    order = np.argsort(sims)[::-1]

    pred = [letters[i] for i in order]

    tfidf_scores.append(
        apk(row["answer"],pred)
    )

    tfidf_correct.append(pred[:3])

# MiniLM

mini_scores=[]

for row in train:

    prompt = row["prompt"]

    opts = [
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]

    p = embedder.encode(
        prompt,
        convert_to_tensor=True
    )

    o = embedder.encode(
        opts,
        convert_to_tensor=True
    )

    sims = util.cos_sim(p,o)[0].cpu().numpy()

    order = np.argsort(sims)[::-1]

    pred = [letters[i] for i in order]

    mini_scores.append(
        apk(row["answer"],pred)
    )

    mini_correct.append(pred[:3])

map3 = np.mean(mini_scores)

count = 0

for row,tf,mini in zip(train,tfidf_correct,mini_correct):

    ans = row["answer"]

    if ans not in tf and ans in mini:
        count += 1

print("\nQ10 Answer 1 (MiniLM MAP@3)")
print(map3)

print("\nQ10 Answer 2 (Improved Count)")
print(count)


Q10 Answer 1 (MiniLM MAP@3)
0.4230833333333333

Q10 Answer 2 (Improved Count)
564


### Question - 11

In [13]:
classifier = pipeline(
    "zero-shot-classification"
)

candidate_labels = [
    train[1]["A"],
    train[1]["B"],
    train[1]["C"]
]

result = classifier(
    train[1]["prompt"],
    candidate_labels
)

print("\nQ11 Answer:")
print(round(result["scores"][0],4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Q11 Answer:
0.4575


### Question - 12

In [14]:
result_multi = classifier(
    train[1]["prompt"],
    candidate_labels,
    multi_label=True
)

softmax_sum = sum(result["scores"])

sigmoid_sum = sum(result_multi["scores"])

print("\nQ12 Answer:")
print(abs(softmax_sum-sigmoid_sum))


Q12 Answer:
0.9994903629012697


### Question - 13

In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load FLAN-T5 Small
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Construct the exact prompt
input_text = (
    f"Question: {train[0]['prompt']}. "
    f"Is the correct answer A: {train[0]['A']} "
    f"or B: {train[0]['B']}? "
    f"Answer with just the letter A or B."
)

# Tokenize
inputs = tokenizer(input_text, return_tensors="pt")

# Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

# Decode and print the answer
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Generated Output:")
print(generated_text)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generated Output:
B
